In [11]:
# ==============================
# INSTALL (only once)
# ==============================
!pip install langchain

# ==============================
# IMPORTS
# ==============================
from langchain_core.runnables import RunnableLambda

# ==============================
# DATA
# ==============================
job_skills = ["python", "machine learning", "pandas", "nlp", "sql"]

resume_strong = "3 years Data Science, Python, Machine Learning, NLP, Pandas, SQL"
resume_avg = "1 year experience, Python, Machine Learning"
resume_weak = "Fresher, Excel"

# ==============================
# STEP 1: SKILL EXTRACTION
# ==============================
def extract_skills(resume):
    return resume.lower()

extract_chain = RunnableLambda(extract_skills)

# ==============================
# STEP 2: MATCHING
# ==============================
def match_skills(resume):
    matched = []
    for skill in job_skills:
        if skill in resume:
            matched.append(skill)
    return matched

match_chain = RunnableLambda(match_skills)

# ==============================
# STEP 3: SCORING
# ==============================
def calculate_score(matched):
    score = int((len(matched) / len(job_skills)) * 100)
    return score

score_chain = RunnableLambda(calculate_score)

# ==============================
# STEP 4: EXPLANATION
# ==============================
def explain(data):
    score = data["score"]
    matched = data["matched"]

    if score == 100:
        explanation = "Candidate matches all required skills."
    elif score >= 60:
        explanation = "Candidate has most required skills."
    elif score >= 30:
        explanation = "Candidate has some relevant skills."
    else:
        explanation = "Candidate lacks required skills."

    return {
        "score": score,
        "matched_skills": matched,
        "explanation": explanation
    }

explain_chain = RunnableLambda(explain)

# ==============================
# FULL PIPELINE (LangChain style)
# ==============================
def pipeline(resume):
    extracted = extract_chain.invoke(resume)
    matched = match_chain.invoke(extracted)
    score = score_chain.invoke(matched)

    final = explain_chain.invoke({
        "score": score,
        "matched": matched
    })

    return final

# ==============================
# RUN
# ==============================
print("===== STRONG =====")
print(pipeline(resume_strong))

print("\n===== AVERAGE =====")
print(pipeline(resume_avg))

print("\n===== WEAK =====")
print(pipeline(resume_weak))

===== STRONG =====
{'score': 100, 'matched_skills': ['python', 'machine learning', 'pandas', 'nlp', 'sql'], 'explanation': 'Candidate matches all required skills.'}

===== AVERAGE =====
{'score': 40, 'matched_skills': ['python', 'machine learning'], 'explanation': 'Candidate has some relevant skills.'}

===== WEAK =====
{'score': 0, 'matched_skills': [], 'explanation': 'Candidate lacks required skills.'}
